# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

In [ ]:
# trl — бібліотека для донавчання мовних моделей (SFT, DPO тощо),
# peft — реалізація LoRA та інших методів PEFT, bitsandbytes — 4-бітна квантизація для QLoRA
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
# Colab має застарілу бібліотеку torchao, з якою peft не імпортується; для цього демо вона не потрібна
!pip uninstall -q -y torchao

In [ ]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [ ]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}


In [ ]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [ ]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [ ]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) — на ньому модель вчиться.
- **Валідаційний** (200) — на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) — на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 — на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [ ]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'content': 'Classify the customer message. Reply with JSON only.',
  'role': 'system'},
 {'content': 'I have got to locate hte bills from {{Person Name}}',
  'role': 'user'},
 {'content': '{"category": "INVOICE", "intent": "check_invoice"}',
  'role': 'assistant'}]

## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [ ]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [ ]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [ ]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` — loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` — кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування — 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` — для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` — кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` — змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` — активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше памʼяті.

In [ ]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,                  # діалог із цього датасету після шаблону займає близько 100 токенів
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # ефективний розмір батча: 4 × 2 = 8 прикладів
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову памʼять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # Для QLoRA бібліотека TRL переводить ваги адаптера в bf16, який T4 не підтримує.
    # Повертаємо їх у fp32: змішана точність fp16 потребує, щоб параметри, що тренуються, були у fp32
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова памʼять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити памʼять для наступної.

In [ ]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

In [ ]:
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB

До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (7 outputs left out)

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.362483,0.478074,0.437490,0.894284,4323.000000
20,0.321579,0.142678,0.187792,0.956574,8672.000000
30,0.080964,0.061208,0.072390,0.985308,13032.000000
40,0.058921,0.027627,0.046916,0.990947,17377.000000
50,0.025130,0.011660,0.028334,0.996557,21714.000000
60,0.006402,0.008153,0.018709,0.998110,26070.000000
70,0.011352,0.006851,0.016694,0.998124,30452.000000
80,0.007675,0.005660,0.013887,0.998745,34813.000000
90,0.005210,0.005878,0.013725,0.998755,39116.000000
100,0.015877,0.005778,0.013602,0.998755,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,97.60
"Тривалість, хв",NaN,2.41
"Пікова памʼять GPU, GB",NaN,1.42
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


In [ ]:
run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.993960,0.438033,0.368946,0.905950,4323.000000
20,0.315899,0.134070,0.193178,0.955362,8672.000000
30,0.080254,0.045521,0.050216,0.988424,13032.000000
40,0.049529,0.024917,0.028862,0.990637,17377.000000
50,0.018369,0.017317,0.025965,0.994075,21714.000000
60,0.012748,0.005098,0.014782,0.998434,26070.000000
70,0.012813,0.004575,0.011225,0.997494,30452.000000
80,0.009829,0.004481,0.010476,0.997494,34813.000000
90,0.012962,0.003906,0.010449,0.998117,39116.000000
100,0.009499,0.003793,0.010383,0.998430,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",62.7,99.40
"Точність intent, %",57.4,96.90
"Тривалість, хв",NaN,2.42
"Пікова памʼять GPU, GB",NaN,3.83
Validation loss,NaN,0.00
"Адаптер, MB",NaN,69.78


In [ ]:
run_lora("Qwen/Qwen3-4B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.768650,0.299733,0.306188,0.922797,4323.000000
20,0.202223,0.100694,0.100053,0.962436,8672.000000
30,0.065427,0.029707,0.035287,0.989987,13032.000000
40,0.035191,0.018448,0.026311,0.995031,17377.000000
50,0.014152,0.007060,0.015569,0.998422,21714.000000
60,0.008942,0.003758,0.009406,0.998437,26070.000000
70,0.007434,0.005294,0.009066,0.997804,30452.000000
80,0.012481,0.004801,0.008346,0.998110,34813.000000
90,0.015871,0.003390,0.008236,0.998742,39116.000000
100,0.012152,0.003282,0.008134,0.998742,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",88.1,100.00
"Точність intent, %",72.5,97.90
"Тривалість, хв",NaN,3.84
"Пікова памʼять GPU, GB",NaN,8.75
Validation loss,NaN,0.00
"Адаптер, MB",NaN,132.19


### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [ ]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Точність category, %",37.9,99.70,62.7,99.40,88.1,100.00
"Точність intent, %",25.6,97.60,57.4,96.90,72.5,97.90
"Тривалість, хв",NaN,2.41,NaN,2.42,NaN,3.84
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` — тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` — для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.255744,0.288718,0.251618,0.934379,4323.000000
20,0.176103,0.060587,0.091798,0.985634,8672.000000
30,0.049239,0.038266,0.028003,0.990942,13032.000000
40,0.046789,0.025577,0.022165,0.994027,17377.000000
50,0.026981,0.007719,0.010975,0.998120,21714.000000
60,0.002136,0.004413,0.004073,0.998442,26070.000000
70,0.005817,0.004062,0.003158,0.999375,30452.000000
80,0.012955,0.003829,0.004107,0.998437,34813.000000
90,0.009278,0.003025,0.003515,0.999370,39116.000000
100,0.014485,0.002765,0.003435,0.999370,43449.000000


📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,Qwen3-0.6B LoRA,Qwen3-0.6B повне
"Чистий JSON, %",100.00,100.00
"Валідний JSON, %",100.00,100.00
"Точність category, %",99.70,99.50
"Точність intent, %",97.60,98.30
"Тривалість, хв",2.41,1.72
"Пікова памʼять GPU, GB",1.42,10.80
Validation loss,0.01,0.00
"Адаптер, MB",40.42,NaN


## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [ ]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

README.md                                0.0 MB
adapter_config.json                      0.0 MB
adapter_model.safetensors              132.2 MB
chat_template.jinja                      0.0 MB
tokenizer.json                          11.4 MB
tokenizer_config.json                    0.0 MB
training_args.bin                        0.0 MB

Для порівняння, ваги Qwen3-4B у fp16: ~8 GB


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [ ]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)

# Злиту модель можна зберегти як звичайну: model.save_pretrained("qwen3-4b-merged")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM


Перевіримо злиту модель на власних повідомленнях:

In [ ]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_period"}

📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "cancel_order"}

📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}

📩 I guess, you have shipped to a wrong address
🤖 {"category": "SHIPPING", "intent": "shipping_address"}



In [ ]:
del model
free_memory()

## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб — найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** — 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** — 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 — формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше — у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-бітний формат, розрахований на нормально розподілені ваги
    bnb_4bit_compute_dtype=torch.float16,   # обчислення виконуються у fp16
    bnb_4bit_use_double_quant=True,         # квантизуються також масштаби квантизації: ще трохи менше памʼяті
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Памʼять GPU після завантаження: 2.7 GB


In [ ]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.481938,0.315485,0.297779,0.923437,4323.000000
20,0.196145,0.127621,0.097099,0.957768,8672.000000
30,0.071648,0.026141,0.042751,0.993448,13032.000000
40,0.028994,0.014699,0.022586,0.996564,17377.000000
50,0.015805,0.012899,0.016191,0.995899,21714.000000
60,0.008535,0.005344,0.008388,0.998728,26070.000000
70,0.006563,0.006242,0.007944,0.998110,30452.000000
80,0.003340,0.006342,0.007191,0.997485,34813.000000
90,0.013318,0.005025,0.007266,0.998110,39116.000000
100,0.011482,0.004832,0.007290,0.998422,43449.000000


## 9. Підсумкове порівняння

In [ ]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA,Qwen3-0.6B повне,Qwen3-4B QLoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Точність category, %",37.9,99.70,62.7,99.40,88.1,100.00,99.50,99.80
"Точність intent, %",25.6,97.60,57.4,96.90,72.5,97.90,98.30,97.70
"Тривалість, хв",NaN,2.41,NaN,2.42,NaN,3.84,1.72,5.25
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75,10.80,3.67
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00,0.00,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19,NaN,NaN


**Що варто помітити:**
- LoRA тренує близько 1–2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й памʼяті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і памʼяттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після — лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше памʼяті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

## Відповіді на запитання 0.1 (питання 22–30)

**22. Як довго виконувалися клітинки ноутбуку 03_sft_lora? Чи були проблеми з запуском, які?**

Запуск через «Виконати всі» на GPU Tesla T4 (15,6 ГБ памʼяті) зайняв близько 40 хвилин. Найдовше виконувались клітинки `run_lora`: Qwen3-0.6B — близько 4 хв, Qwen3-1.7B — близько 7 хв, Qwen3-4B — 11 хв (разом із завантаженням моделі, двома оцінками на 1000 прикладах і самим тренуванням). Саме тренування (100 кроків) тривало 2,41; 2,42 і 3,84 хв відповідно. Повне донавчання Qwen3-0.6B — 1,72 хв, QLoRA для Qwen3-4B — 5,25 хв. Помилок виконання не було, виводились лише попередження бібліотек: про відсутність `HF_TOKEN` (запити до Hugging Face Hub без авторизації) і про `base_model_name_or_path` у конфігурації PEFT; на результат вони не впливають. Окрема проблема була не з кодом: наприкінці Colab показав «Не вдалось автоматично зберегти» через те, що файл на Диску оновлено в іншій вкладці; зміни довелося зберегти вручну через «Перегляд змін».

**23. Прокоментуйте вивід «Памʼять GPU після завантаження». Чи очікуваним для Вас є результат?**

Після завантаження в fp16: Qwen3-0.6B — 1,2 GB, Qwen3-1.7B — 3,5 GB, Qwen3-4B — 8,1 GB. Результат очікуваний: у fp16 кожен параметр займає 2 байти, тож памʼять ≈ кількість параметрів × 2 (0,60 млрд → 1,2 GB; 1,72 млрд → ≈3,4 GB; 4,02 млрд → ≈8,0 GB). Збіг майже точний, бо після завантаження в памʼяті лише ваги, без градієнтів, станів оптимізатора й активацій. Для QLoRA ті самі ваги Qwen3-4B у 4 бітах займають 2,7 GB, тобто втричі менше, ніж у fp16 (8,1 GB). Це трохи більше за теоретичні 0,5 байта на параметр (≈2 GB): до 4-бітних ваг додаються масштаби блоків, а частина ваг (імовірно, матриця ембедингів і вихідний шар) лишається не квантизованою.

**24. Що робить рядок `with torch.no_grad()` у функції `classify()`?**

Він вимикає відстеження градієнтів усередині блоку. Під час інференсу ми лише обчислюємо відповідь, зворотного проходу немає, тому PyTorch не має зберігати проміжні результати для градієнтів. Це зменшує споживання памʼяті GPU (особливо для батчів по 32 звернення) і трохи прискорює генерацію. На самі відповіді моделі це не впливає.

**25. Для чого у `classify()` задано `do_sample=False`?**

Це жадібна генерація (greedy decoding): на кожному кроці береться токен із найбільшою ймовірністю, випадковості немає. Для класифікації це потрібно: ми хочемо одну найімовірнішу відповідь у строгому форматі JSON, а не різноманітність. З семплінгом одне й те саме звернення могло б щоразу отримувати різні мітки, метрики були б нестабільними й невідтворюваними, а порівняння моделей до і після донавчання — нечесним. Параметр `temperature` при цьому не потрібен.

**26. В демо 1 було натреновано класифікатор з базової моделі за допомогою заміни голови. В демо 3 в ході інференсу моделі також відбувається класифікація звернень. Чому не можна застосувати такий же підхід?**

У демо 1 BERT — це encoder: він перетворює текст на вектори, а ми додали нову голову (лінійний шар), яка видає номер одного класу з фіксованого списку. В демо 3 моделі Qwen3 — генеративні (decoder), і відповідь має бути текстом: JSON одразу з двома полями, `category` (11 значень) і `intent` (27 значень). Заміна голови тут не підходить: вона дала б лише індекс одного класу (для двох міток довелося б робити дві голови або склеювати мітки), голова ініціалізується випадково й вчиться з нуля, а модель перестає генерувати текст, тобто втрачає універсальність, і нові формати відповіді чи нові мітки вимагали б змінювати архітектуру. SFT (разом з LoRA) не змінює архітектуру моделі: ми вчимо її відповідати у потрібному форматі, зберігаючи інші можливості, а тренуються лише невеликі адаптери.

**27. Порівняйте кількість токенів короткого й детального промпту (розділ 2). Як ця різниця вплине на вартість обробки 10 000 звернень на день?**

Короткий промпт — 11 токенів, детальний — 166 токенів, тобто різниця 155 токенів на кожен запит (детальний у 15 разів довший). Системний промпт передається з кожним зверненням: для 10 000 звернень на день детальний дає 166 × 10 000 ≈ 1,66 млн токенів лише на інструкцію, короткий — 0,11 млн. Економія — близько 1,55 млн вхідних токенів на день (≈46 млн на місяць). Оскільки текст самого звернення короткий (десятки токенів), загальна довжина запиту після донавчання зменшується в кілька разів, тож пропорційно знижується вартість обробки (за оплати за токен) або навантаження на GPU й затримка (на власному сервері). Донавчена модель при цьому точніша, бо перелік міток зберігається у вагах, а не в промпті.

**28. Для чого в датасеті задано колонку `chat_template_kwargs` з `enable_thinking=False`? Що, на Вашу думку, сталося б із моделлю, якби її донавчали на цих прикладах у режимі `/think`?**

Колонка передає в шаблон розмови Qwen3 параметр `enable_thinking=False`: шаблон додає до відповіді асистента порожній блок `<think></think>`, і модель відповідає JSON одразу, без міркувань (як у режимі `/no_think` у демо 2). Якби донавчати на цих прикладах у режимі `/think`, виникла б така сама розбіжність між шаблоном і даними, як у прикладі з GSM8K у демо 2: у даних відповідь складається лише з JSON, без міркувань, а шаблон у режимі міркування очікує блок `<think>…</think>` перед відповіддю. Модель могла б навчитися пропускати міркування, і режим `/think` фактично перестав би працювати, а в разі міркувань перед JSON відповіді стали б довшими й повільнішими, не вміщувалися б у `max_new_tokens=60` і порушували б перевірку «чистий JSON». Нам потрібна швидка відповідь суворо у форматі JSON, тому міркування вимкнено і в даних, і під час інференсу (`enable_thinking=False` у `classify_batch`).

**29. Що означає параметр `assistant_only_loss=True`? На яких токенах модель вчилася б без нього і чим це погано для нашої задачі?**

Параметр означає, що loss рахується лише на токенах відповіді асистента (JSON з мітками), а токени системного промпту й повідомлення клієнта отримують мітку −100, і функція втрат їх пропускає. Без нього модель вчилася б передбачати всі токени діалогу: системну інструкцію, текст звернення клієнта та відповідь. Для нашої задачі це погано: з приблизно 100 токенів діалогу корисна частина (JSON з мітками) становить лише близько 15–20 токенів, тож більша частина навчання йшла б на запамʼятовування однакового системного промпту й відтворення чужих повідомлень клієнтів, а не на зіставлення звернення з категорією та наміром. Значення loss розмивалося б і здавалося кращим, ніж реальна якість класифікації.

**30. Порівняйте LoRA і QLoRA за пам'яттю, часом тренування і якістю (розділ 9). У яких випадках Ви обрали б QLoRA?**

Для Qwen3-4B (розділ 9): пікова памʼять GPU — 8,75 GB для LoRA проти 3,67 GB для QLoRA (у 2,4 раза менше); час тренування — 3,84 хв проти 5,25 хв (QLoRA на 37% повільніша, бо ваги щоразу деквантизуються під час обчислень); якість майже однакова: точність `category` 100,00% проти 99,80%, точність `intent` 97,90% проти 97,70% (різниця 0,2 п.п. — це 2 звернення з 1000, у межах похибки), validation loss в обох близький до 0. QLoRA я обрав би тоді, коли памʼяті GPU не вистачає для LoRA в fp16: наприклад, модель на 7–8 млрд параметрів (≈14–16 GB ваг у fp16) не вміститься на T4, а в 4 бітах вміститься; також для тренування на безкоштовному T4 чи ноутбуці з малою памʼяттю. Ціна — приблизно на третину більший час і мінімальна втрата якості. Якщо памʼяті достатньо (як для Qwen3-4B: 8,75 GB із 15,6 GB), краще LoRA: вона швидша.